<a href="https://colab.research.google.com/github/kritikaamohan/Text-Classification-Pipeline/blob/main/textclasshugface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this project i am gonna use **20 Newsgroups dataset** which is a well-know text classification dataset containing ~18000 newsgroups documents categorized into 20 topics.

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [3]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from transformers import TrainingArguments, Trainer
import numpy as np

In [4]:
#loading dataset
newsgroups= fetch_20newsgroups(subset='all', remove=('headers','footers','quotes'))

texts= newsgroups.data
labels= newsgroups.target
label_names = newsgroups.target_names

In [5]:
# 1. How many examples
print("Number of examples:", len(texts))

# 2. Look at one raw text
print("\nFirst text:\n", texts[0])

# 3. Its label (just a number)
print("\nLabels number:", labels[0])

# 4. Translate label number to category name
print("\nCategory name:", label_names[labels[0]])

# 5. Class balance
from collections import Counter
counts = Counter(labels)
for label_num, count in counts.items():
    print(label_names[label_num], ":", count)

Number of examples: 18846

First text:
 

I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a lot of
fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final
regular season game.          PENS RULE!!!



Labels number: 10

Category name: rec.sport.hockey
rec.sport.hockey : 999
comp.sys.ibm.pc.hardware : 982
talk.politics.mideast : 940
comp.sys.mac.hardware : 963
sci.electronics : 984
talk.religion.misc : 628
sci.crypt : 991
sci.med : 990
alt.athe

In [6]:
# split into train and test
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42)
print(train_texts[0])
print("~~~~~~")
print(test_texts[0])
print("~~~~~~")
print(train_labels)
print("~~~~~~")
print(test_labels)

#
# I've gotten very few posts on this group in the last couple days.  (I
# recently added it to my feed list.)  Is it just me, or is this group
# near death?
#

Seen from the mailing list side, I'm getting about the right amount of
traffic.

Patrick L. Mahan

--- TGV Window Washer ------------------------------- Mahan@TGV.COM ---------

Waking a person unnecessarily should not be considered  - Lazarus Long
a capital crime.  For a first offense, that is            From the Notebooks of
							  Lazarus Long

Patrick L. Mahan

--- TGV Window Washer ------------------------------- Mahan@TGV.COM ---------
~~~~~~



	The runner can leave his base at any time.  If the ball is caught,
he's got to tag up.  If it isn't caught, he _doesn't_ have to tag up at
all.  So, if he's feeling lucky, your runner at second can sprint for glory
as soon as the ball is popped up.  If it isn't caught, he's probably scored
a run.  If it is, he's probably headed for AAA.  

	The only effect the infield fly has 

Tokenization with hugging face. i will convert the raw text into a numerical format that the model (tokenization) understands. Trandormer moels use subword tokenization to break text into smaller, more meaningful units. I will use DistilBERT transformer which is smaller and faster and one of most well-known tranformer language models.

In [7]:
#loading tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

#tokenizing the text data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

This tokenizer handles lowercasing, padding, and truncating, converting each input text into input_ids and attention_mask.

after tokenizing our text data, the next step is to structure it in a format suitable for training with pytorch and hugging face's trainer api.

Transformer models expect inputs like input_ids, attention_mask, and labels to be provides as tensors.
To streamline this process and allow efficient data loading during training, i will create a custom dataset class by subclassing torch.utils.data.Dataset.

In [8]:
class NewsGroupDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} | {'labels': torch.tensor(self.labels[idx])}

train_dataset = NewsGroupDataset(train_encodings, train_labels)
test_dataset = NewsGroupDataset(test_encodings, test_labels)

this class will take the tokenized encodings and corresponding labels and return them in the format the model expects.

With our dataset now properly tokenized and formatted, it’s time to select and load a pre-trained transformer model for our classification task. Instead of training a model from scratch, which requires massive data and compute, we leverage AutoModelForSequenceClassification, which wraps pre-trained models like BERT, DistilBERT, and RoBERTa specifically for classification problems. In this case, we’ll use distilbert-base-uncased, a lightweight version of BERT that’s faster and still highly effective

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=20)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Since our dataset has 20 unique categories, we set num_labels=20 to adapt the model’s final classification layer accordingly.

# Define Training Configuration
Before we can train our model, we need to define the training configuration, which includes how the model should learn, how often to evaluate, when to save checkpoints, and other essential hyperparameters.


Hugging Face provides a convenient TrainingArguments class that lets us configure all of this in one place. We’ll specify parameters such as the learning rate, batch sizes, number of training epochs, weight decay for regularization, and logging frequency. Additionally, we will define a compute_metrics function to evaluate our model using accuracy during training

In [10]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds)}

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_strategy="epoch",
    eval_steps=500
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


This setup ensures our model is trained efficiently and consistently assessed throughout the process.

# Train the Model
With our model, datasets, training arguments, and evaluation metrics all defined, we’re ready to bring everything together using Hugging Face’s Trainer API.

The Trainer class abstracts away much of the boilerplate involved in training a transformer model, handling batching, optimization, evaluation, logging, and checkpointing behind the scenes. By passing in our model, training configuration, datasets, and metric function, we can initiate the fine-tuning process in a single line of code. It enables experimentation to be faster, cleaner, and easier to scale.

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()

Step,Training Loss
100,2.602169
200,1.852618
300,1.567912
400,1.411285


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=472, training_loss=1.7851101341894118, metrics={'train_runtime': 371.0764, 'train_samples_per_second': 40.628, 'train_steps_per_second': 1.272, 'total_flos': 998859786608640.0, 'train_loss': 1.7851101341894118, 'epoch': 1.0})

In [12]:
metrics = trainer.evaluate()
print(metrics)

Training Loss,Validation Loss,Step,Accuracy
1.411285,1.351251,472,0.650133


{'eval_loss': 1.3512510061264038, 'eval_accuracy': 0.650132625994695}


Now that our model is trained, the final step is to use it for making predictions on new, unseen text. This process is called inference. In a real-world scenario, you might receive raw text data, like an email, a news headline, or a customer message, and you want the model to classify it into one of the predefined categories.

To do this, we first tokenize the input text using the same tokenizer used during training, convert it into tensors, and pass it through the model.

In [15]:
text = "The government passed a new law affecting international trade."

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}   # <-- add this line

outputs = model(**inputs)
predicted_class = outputs.logits.argmax().item()

print(f"Predicted Topic: {label_names[predicted_class]}")

Predicted Topic: talk.politics.mideast


The model outputs a set of logits (unnormalized predictions), and we select the index with the highest score as the predicted class.

Text classification with Hugging Face Transformers offers a powerful and efficient approach for transforming raw text into actionable insights by leveraging state-of-the-art NLP models. By following this pipeline, from loading data and tokenization to model training and inference, you now have a solid foundation for building, fine-tuning, and deploying real-world NLP solutions. Whether you’re classifying emails, reviews, or news articles, this approach scales effortlessly and sets you up for production-ready applications.

In [16]:
# --- Reusable classification function ---
def classify(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model(**inputs)
    predicted_class = outputs.logits.argmax().item()
    return label_names[predicted_class]


# --- Test on your own made-up sentences ---
texts_to_check = [
    "The government passed a new law affecting international trade.",
    "The new graphics card doubles the frame rate in most games.",
    "The team won the championship after a dramatic overtime finish.",
    "Doctors recommend regular exercise to improve heart health.",
    "The spacecraft successfully landed on the moon's surface.",
]

print("--- Predictions on custom text ---\n")
for t in texts_to_check:
    print(f"Text: {t}")
    print(f"Predicted: {classify(t)}\n")


# --- Test on real examples from the test set, with true labels ---
print("--- Predictions on real test set examples ---\n")
for i in range(5):
    text = test_texts[i]
    true_label = label_names[test_labels[i]]
    predicted_label = classify(text)
    correct = "✓" if true_label == predicted_label else "✗"
    print(f"{correct} True: {true_label} | Predicted: {predicted_label}")
    print(f"Text snippet: {text[:100]}...\n")

--- Predictions on custom text ---

Text: The government passed a new law affecting international trade.
Predicted: talk.politics.mideast

Text: The new graphics card doubles the frame rate in most games.
Predicted: comp.sys.ibm.pc.hardware

Text: The team won the championship after a dramatic overtime finish.
Predicted: rec.sport.hockey

Text: Doctors recommend regular exercise to improve heart health.
Predicted: sci.med

Text: The spacecraft successfully landed on the moon's surface.
Predicted: sci.space

--- Predictions on real test set examples ---

✓ True: rec.sport.baseball | Predicted: rec.sport.baseball
Text snippet: 


	The runner can leave his base at any time.  If the ball is caught,
he's got to tag up.  If it is...

✗ True: sci.electronics | Predicted: comp.windows.x
Text snippet: 
Well, it's not an FTP site, but I got an 800 number for Signetics BBS.

The Signetics BBS contain s...

✓ True: sci.space | Predicted: sci.space
Text snippet: Hi,
    I was reading through "The S